# Lab 1 - Exercise 2: Pipeline Consolidation

Questo notebook continua direttamente dall'Exercise 1.3. Nel notebook precedente abbiamo verificato che una ResNet-18 con backbone congelato e sola testa finale addestrabile e' una baseline utile, ma non sufficiente a superare la baseline SVM.

Qui l'obiettivo cambia: non vogliamo accumulare nuove celle di training ripetitive, ma consolidare una pipeline riproducibile per lanciare, registrare e confrontare esperimenti.

---
---
## Exercise 2: Pipeline Consolidation

Consolidate your implementation. When building applications based on Deep Learning, you will inevitably need to run many experiments. So, it is always a good idea to engineer a reproducible pipeline that allows you to run and re-run experiments with different hyperparameters.

Aspetti richiesti:

- **Model and Loss abstraction**: la pipeline deve poter cambiare modello, loss e optimizer senza riscrivere il training loop.
- **Configuration management**: gli iperparametri non devono essere sparsi nelle celle, ma raccolti in una configurazione.
- **Logging**: le run devono produrre metriche salvabili e confrontabili; WandB puo' essere usato, ma resta opzionale.

## Collegamento con Exercise 1.3

La run head-only appena conclusa ha ottenuto circa **0.478** di validation accuracy con split anti-leakage. Il risultato e' piu' basso dello screening vecchio, ma piu' affidabile perche' lo split separa gruppi consecutivi di immagini simili.

Questo e' esattamente il motivo per cui serve una pipeline consolidata: quando cambiano split, batch size, optimizer o loss, dobbiamo poter ricostruire chiaramente cosa e' stato eseguito e confrontare i risultati in modo ordinato.

In [10]:
from pathlib import Path
import sys

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from dla_lab1.config import experiment_config, load_config
from dla_lab1.experiments import batch_size_for, run_finetuning
from dla_lab1.tracking import (
    experiment_output_dir,
    save_run_artifacts,
    summarize_history,
    wandb_is_available,
)

config = load_config(ROOT / "config" / "config.yaml")

# Lasciato False per evitare di rilanciare training lunghi quando si esegue il notebook.
RUN_TRAINING = True

# WandB e' opzionale. Attivalo solo dopo aver installato wandb e fatto login.
ENABLE_WANDB = False
config["wandb"]["enabled"] = ENABLE_WANDB

print(f"Project root: {ROOT}")
print(f"WandB installed: {wandb_is_available()}")
print(f"WandB enabled for this notebook: {config['wandb']['enabled']}")

Project root: c:\Users\checc\OneDrive\Desktop\DLA\DLA_Lab\DLA_1
WandB installed: True
WandB enabled for this notebook: False


## Cosa era stato fatto nel vecchio notebook

Nel vecchio notebook l'idea principale era corretta: era stata creata una funzione `train_and_evaluate` e poi erano stati lanciati diversi esperimenti cambiando loss e optimizer.

Il limite era la forma: molte configurazioni erano scritte direttamente nelle celle, alcune celle erano quasi duplicate e WandB era gestito manualmente nel notebook. Nel refactoring, questa logica viene spostata in `src/dla_lab1` e controllata da `config/config.yaml`.

In [2]:
old_screening_results = pd.DataFrame([
    ["Adam_CrossEntropy", "CrossEntropy", "Adam", 0.5273, 8],
    ["AdamW_CrossEntropy", "CrossEntropy", "AdamW", 0.5240, 5],
    ["SGD_CrossEntropy", "CrossEntropy", "SGD", 0.4463, 9],
    ["Adam_WeightedCrossEntropy", "WeightedCrossEntropy", "Adam", 0.5139, 7],
    ["AdamW_WeightedCrossEntropy", "WeightedCrossEntropy", "AdamW", 0.5135, 10],
    ["SGD_WeightedCrossEntropy", "WeightedCrossEntropy", "SGD", 0.3552, 10],
    ["Adam_FocalLoss", "FocalLoss", "Adam", 0.4860, 6],
    ["AdamW_FocalLoss", "FocalLoss", "AdamW", 0.4925, 10],
    ["SGD_FocalLoss", "FocalLoss", "SGD", 0.3322, 10],
], columns=["Experiment", "Loss", "Optimizer", "Best Val Accuracy", "Best Epoch"])

old_screening_results.sort_values("Best Val Accuracy", ascending=False)

,Experiment,Loss,Optimizer,Best Val Accuracy,Best Epoch
0,Adam_CrossEntropy,CrossEntropy,Adam,0.5273,8
1,AdamW_CrossEntropy,CrossEntropy,AdamW,0.5240,5
3,Adam_WeightedCrossEntropy,WeightedCrossEntropy,Adam,0.5139,7
4,AdamW_WeightedCrossEntropy,WeightedCrossEntropy,AdamW,0.5135,10
7,AdamW_FocalLoss,FocalLoss,AdamW,0.4925,10
6,Adam_FocalLoss,FocalLoss,Adam,0.4860,6
2,SGD_CrossEntropy,CrossEntropy,SGD,0.4463,9
5,SGD_WeightedCrossEntropy,WeightedCrossEntropy,SGD,0.3552,10
8,SGD_FocalLoss,FocalLoss,SGD,0.3322,10


### Commento sullo screening vecchio

Gli esperimenti vecchi sono trattati come screening preliminare. Non li rilanciamo tutti perche' costerebbe molto tempo e produrrebbe un notebook ripetitivo.

L'informazione utile e' che, nel setup head-only, la combinazione piu' efficace era `Adam + CrossEntropy`. WeightedCrossEntropy e FocalLoss non hanno migliorato la performance, quindi lo sbilanciamento delle classi non era l'unico limite: il problema principale era il backbone congelato.

## Nuova architettura della pipeline

La pipeline e' divisa in moduli piccoli. Ogni funzione importante e' commentata nel relativo file `.py`, cosi' il notebook resta leggibile e non diventa un blocco di implementazione.

In [3]:
pipeline_components = pd.DataFrame([
    ["config.py", "load_config, experiment_config", "Legge il YAML e costruisce la configurazione finale della run."],
    ["data.py", "build_dataloaders", "Crea train/validation/test loader e split anti-leakage."],
    ["models.py", "build_classifier", "Crea ResNet-18 e sostituisce la testa finale per 43 classi."],
    ["losses.py", "build_loss", "Seleziona CrossEntropy, WeightedCrossEntropy o FocalLoss."],
    ["train.py", "build_optimizer, train_model", "Crea optimizer/scheduler e gestisce training, validation, checkpoint."],
    ["evaluate.py", "predict, classification_metrics", "Calcola predizioni e metriche di classificazione."],
    ["tracking.py", "save_run_artifacts", "Salva history, config e summary della run in artifacts/runs."],
], columns=["File", "Funzioni principali", "Utilizzo"])

pipeline_components

,File,Funzioni principali,Utilizzo
0,config.py,"load_config, experiment_config",Legge il YAML e costruisce la configurazione f...
1,data.py,build_dataloaders,Crea train/validation/test loader e split anti...
2,models.py,build_classifier,Crea ResNet-18 e sostituisce la testa finale p...
3,losses.py,build_loss,"Seleziona CrossEntropy, WeightedCrossEntropy o..."
4,train.py,"build_optimizer, train_model","Crea optimizer/scheduler e gestisce training, ..."
5,evaluate.py,"predict, classification_metrics",Calcola predizioni e metriche di classificazione.
6,tracking.py,save_run_artifacts,"Salva history, config e summary della run in a..."


## Configuration management

Gli iperparametri principali sono centralizzati in `config/config.yaml`.

Questo rende piu' semplice:

- cambiare batch size;
- cambiare optimizer;
- cambiare loss;
- abilitare o disabilitare WandB;
- rilanciare una run con gli stessi parametri.

In [4]:
available_experiments = pd.DataFrame([
    [name, exp.get("description", ""), exp.get("batch_size_key", "")]
    for name, exp in config["experiments"].items()
], columns=["experiment_name", "description", "batch_size_key"])

available_experiments

,experiment_name,description,batch_size_key
0,feature_svm,ResNet-18 feature extractor plus linear SVM ba...,batch_size_feature_extraction
1,finetune_frozen,Fine-tune only the final classifier head.,batch_size_finetune_frozen
2,ex1_3_head_only_adam_ce,Exercise 1.3 head-only fine-tuning baseline se...,batch_size_head_only_original
3,finetune_layer4,Fine-tune classifier head and ResNet-18 layer4.,batch_size_finetune_layer4


In [5]:
selected_experiment = "ex1_3_head_only_adam_ce"
exp_cfg = experiment_config(config, selected_experiment)
batch_size = batch_size_for(config, exp_cfg["experiment"]["batch_size_key"])

selected_config_summary = pd.DataFrame([
    ["experiment", selected_experiment],
    ["model", exp_cfg["model"]["name"]],
    ["freeze_backbone", exp_cfg["model"]["freeze_backbone"]],
    ["unfreeze_layer4", exp_cfg["model"].get("unfreeze_layer4", False)],
    ["loss", exp_cfg["training"]["loss"]],
    ["optimizer", exp_cfg["training"]["optimizer"]],
    ["learning_rate", exp_cfg["training"]["learning_rate"]],
    ["batch_size", batch_size],
    ["epochs", exp_cfg["training"]["epochs"]],
    ["scheduler", exp_cfg["training"]["scheduler"]],
], columns=["parameter", "value"])

selected_config_summary

,parameter,value
0,experiment,ex1_3_head_only_adam_ce
1,model,resnet18
2,freeze_backbone,True
3,unfreeze_layer4,False
4,loss,CrossEntropy
5,optimizer,Adam
6,learning_rate,0.001
7,batch_size,128
8,epochs,10
9,scheduler,step


## Logging locale e WandB

Il logging locale e' sempre disponibile e salva i risultati in:

`artifacts/runs/<experiment_name>/`

Dentro questa cartella vengono salvati:

- `history.csv`: loss, accuracy e learning rate per epoca;
- `config_used.yaml`: configurazione usata nella run;
- `summary.json`: riassunto con best epoch e best validation accuracy.

WandB e' supportato dalla pipeline, ma non e' obbligatorio. Per usarlo bisogna installare `wandb`, fare login e impostare `ENABLE_WANDB = True`.

In [6]:
wandb_status = pd.DataFrame([
    ["wandb_installed", wandb_is_available()],
    ["wandb_enabled", config["wandb"]["enabled"]],
    ["wandb_project", config["wandb"]["project"]],
    ["wandb_group", config["wandb"]["group"]],
    ["local_output_dir", experiment_output_dir(config, selected_experiment, root=ROOT)],
], columns=["item", "value"])

wandb_status

,item,value
0,wandb_installed,True
1,wandb_enabled,False
2,wandb_project,DLA_Lab1_Pipeline
3,wandb_group,exercise_2_pipeline_consolidation
4,local_output_dir,c:\Users\checc\OneDrive\Desktop\DLA\DLA_Lab\DL...


## Risultato documentato dell'Exercise 1.3

Per rendere il notebook leggero, qui riportiamo la history ottenuta nella run precedente invece di rilanciare automaticamente 10 epoche. Questo e' coerente con l'Exercise 2: l'obiettivo e' consolidare e confrontare gli esperimenti, non duplicare training lunghi.

In [7]:
exercise_13_history = pd.DataFrame([
    [1, 1.744337, 0.517483, 2.087210, 0.429725, 0.00100],
    [2, 0.988288, 0.711191, 2.060829, 0.452577, 0.00100],
    [3, 0.818380, 0.752498, 2.083187, 0.457732, 0.00100],
    [4, 0.734424, 0.777233, 2.193421, 0.451890, 0.00100],
    [5, 0.633942, 0.813160, 2.085414, 0.478179, 0.00010],
    [6, 0.624741, 0.815466, 2.089019, 0.472680, 0.00010],
    [7, 0.624248, 0.815226, 2.111028, 0.469588, 0.00010],
    [8, 0.618714, 0.816907, 2.106734, 0.474914, 0.00010],
    [9, 0.603214, 0.821134, 2.101801, 0.474570, 0.00001],
    [10, 0.604205, 0.822430, 2.085159, 0.473540, 0.00001],
], columns=["epoch", "train_loss", "train_acc", "val_loss", "val_acc", "learning_rate"])

exercise_13_history

,epoch,train_loss,train_acc,val_loss,val_acc,learning_rate
0,1,1.744337,0.517483,2.087210,0.429725,0.00100
1,2,0.988288,0.711191,2.060829,0.452577,0.00100
2,3,0.818380,0.752498,2.083187,0.457732,0.00100
3,4,0.734424,0.777233,2.193421,0.451890,0.00100
4,5,0.633942,0.813160,2.085414,0.478179,0.00010
5,6,0.624741,0.815466,2.089019,0.472680,0.00010
6,7,0.624248,0.815226,2.111028,0.469588,0.00010
7,8,0.618714,0.816907,2.106734,0.474914,0.00010
8,9,0.603214,0.821134,2.101801,0.474570,0.00001
9,10,0.604205,0.822430,2.085159,0.473540,0.00001


In [8]:
exercise_13_summary = summarize_history(exercise_13_history.to_dict("records"))
pd.DataFrame([exercise_13_summary])

,best_epoch,best_val_acc,best_val_loss,last_epoch,last_train_acc,last_val_acc
0,5,0.478179,2.085414,10,0.82243,0.47354


### Commento sul risultato

Il miglior checkpoint dell'Exercise 1.3 arriva alla quinta epoca, con validation accuracy circa **0.478**. La training accuracy sale oltre **0.82**, mentre la validation resta molto piu' bassa: questo indica che la testa finale impara il training set, ma il backbone congelato limita la generalizzazione.

La differenza rispetto allo screening vecchio, dove `Adam + CrossEntropy` arrivava a **0.5273**, dipende soprattutto dallo split anti-leakage piu' severo usato ora.

## Cella opzionale per rilanciare una run

La cella seguente e' disattivata di default. Per usarla:

1. imposta `RUN_TRAINING = True`;
2. se vuoi WandB, installa `wandb`, fai login e imposta `ENABLE_WANDB = True`;
3. scegli `experiment_to_run`.

La funzione `run_finetuning` salva automaticamente checkpoint e artifact locali. Se WandB e' abilitato, invia anche le metriche online nel progetto/gruppo indicato dal config.

In [11]:
experiment_to_run = "ex1_3_head_only_adam_ce"

if RUN_TRAINING:
    config["wandb"]["enabled"] = ENABLE_WANDB
    result = run_finetuning(config, experiment_to_run, root=ROOT)
    run_summary = result["artifacts"]["summary"]
    print(f"Run salvata in: {result['artifacts']['output_dir']}")
    pd.DataFrame([run_summary])
else:
    print("Training non eseguito. Imposta RUN_TRAINING = True per rilanciare la run.")

  0%|          | 0/163 [00:09<?, ?it/s]

KeyboardInterrupt: 

## Conclusione Exercise 2

La pipeline ora soddisfa le richieste dell'esercizio:

- il modello e' astratto in `build_classifier`;
- loss e optimizer sono selezionati da configurazione;
- gli iperparametri sono centralizzati in `config.yaml`;
- il training loop e' riutilizzabile;
- i risultati sono salvati localmente in modo riproducibile;
- WandB e' predisposto come logging opzionale, senza API key nel notebook.

Il notebook non e' piu' una sequenza di celle duplicate: diventa un'interfaccia pulita per configurare, lanciare e confrontare esperimenti.